# Day 8: Data quality checks for football data

**Phase 1: Foundations & Infrastructure**

## Objective
Data quality checks for football data

## Task
- Executed high-level profiling of StatsBomb event data for the 2022 World Cup Final (Argentina vs France).
- Verified Primary Key integrity (duplicate check) and Referential Integrity (match_id alignment).
- Audited critical null values across core event features (player, location, type).
- Validated spatial boundaries (0-120x, 0-80y) and Expected Goals (xG) distribution consistency.
- Implemented a three-tier advanced cleaning strategy:
    - **Strategy 1:** Imputed implicit boolean 'False' values for sparse indicators (e.g., under_pressure).
    - **Strategy 2:** Optimized memory by dropping features with >98% null density.
    - **Strategy 3:** Verified contextual integrity for specialized metrics like pass_length.
- Generated an automated, professional HTML profiling report using ydata-profiling saved in reports/html/.

**Deliverable:** Notebook day008_football_data_quality.ipynb + HTML profiling report.

---

### Instructions:
*Treat this as your course workbook. Write your code, notes, or execution steps below.*

Remember: 
- Document your decisions.
- Use clear variable names.
- If today's task involves creating a script in `src/`, a dashboard in `apps/`, or documentation in `docs/`, you can use this notebook to test your logic or simply write your reflections and proofs of execution (e.g. running terminal commands with `!`).



## Chech which id match we will use

In [13]:
from statsbombpy import sb

# 1. Get all available free competitions
print("--- Step 1: Find Competition and Season IDs ---")
comps = sb.competitions()

# Display just the essential columns to find what we need
display(comps[['competition_id', 'season_id', 'competition_name', 'season_name', 'match_updated']])

--- Step 1: Find Competition and Season IDs ---


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,competition_id,season_id,competition_name,season_name,match_updated
0,9,281,1. Bundesliga,2023/2024,2024-09-28T20:46:38.893391
1,9,27,1. Bundesliga,2015/2016,2024-05-19T11:11:14.192381
2,1267,107,African Cup of Nations,2023,2024-09-28T01:57:35.846538
3,16,4,Champions League,2018/2019,2025-05-08T15:10:50.835274
4,16,1,Champions League,2017/2018,2024-02-13T02:35:28.134882
...,...,...,...,...,...
70,35,75,UEFA Europa League,1988/1989,2024-02-12T14:45:05.702250
71,53,315,UEFA Women's Euro,2025,2025-07-28T14:19:20.467348
72,53,106,UEFA Women's Euro,2022,2024-02-13T13:27:17.178263
73,72,107,Women's World Cup,2023,2025-07-14T10:07:06.620906


In [14]:

# ---------------------------------------------------------
# Let's say from the table above, you spotted the 2022 FIFA World Cup.
# The 'competition_id' is 43 and 'season_id' is 106.
# ---------------------------------------------------------
# 2. Get all matches for that specific competition and season
print("\n--- Step 2: Get all matches for the selected tournament ---")
matches = sb.matches(competition_id=43, season_id=106)

# Display the matches to find the specific game
display(matches[['match_id', 'match_date', 'home_team', 'away_team', 'home_score', 'away_score']].head())

# 3. Optional: Filter programmatically if you know the teams
print("\n--- Step 3: Search for a specific team's match ---")
my_team_search = matches[(matches['home_team'] == 'Argentina') | (matches['away_team'] == 'Argentina')]
display(my_team_search[['match_id', 'match_date', 'home_team', 'away_team']])


--- Step 2: Get all matches for the selected tournament ---


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,match_id,match_date,home_team,away_team,home_score,away_score
0,3857256,2022-12-02,Serbia,Switzerland,2,3
1,3869151,2022-12-03,Argentina,Australia,2,1
2,3857257,2022-11-30,Australia,Denmark,1,0
3,3857258,2022-11-24,Brazil,Serbia,2,0
4,3857288,2022-11-26,Tunisia,Australia,0,1



--- Step 3: Search for a specific team's match ---


,match_id,match_date,home_team,away_team
1,3869151,2022-12-03,Argentina,Australia
6,3869321,2022-12-09,Netherlands,Argentina
9,3869685,2022-12-18,Argentina,France
11,3857264,2022-11-30,Poland,Argentina
13,3857289,2022-11-26,Argentina,Mexico
19,3869519,2022-12-13,Argentina,Croatia
37,3857300,2022-11-22,Argentina,Saudi Arabia


In [ ]:
%pip install ydata-profiling

In [16]:
import pandas as pd
import numpy as np
from statsbombpy import sb
from ydata_profiling import ProfileReport

# Load events for a specific match (e.g., 2022 World Cup Final: Argentina vs France)
match_id = 3869685
print(f"Loading events for match {match_id}...")

events_df = sb.events(match_id=match_id)
display(events_df.head(5))


Loading events for match 3869685...


/Library/Frameworks/Python.framework/Versions/3.13/lib/python3.13/site-packages/statsbombpy/api_client.py:21: NoAuthWarning: credentials were not supplied. open data access only
  warnings.warn(


,50_50,bad_behaviour_card,ball_receipt_outcome,ball_recovery_offensive,ball_recovery_recovery_failure,block_deflection,block_offensive,carry_end_location,clearance_aerial_won,clearance_body_part,...,substitution_outcome,substitution_outcome_id,substitution_replacement,substitution_replacement_id,tactics,team,team_id,timestamp,type,under_pressure
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,"{'formation': 433, 'lineup': [{'player': {'id'...",Argentina,779,00:00:00.000,Starting XI,NaN
1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,"{'formation': 4231, 'lineup': [{'player': {'id...",France,771,00:00:00.000,Starting XI,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,France,771,00:00:00.000,Half Start,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,Argentina,779,00:00:00.000,Half Start,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,France,771,00:00:00.000,Half Start,NaN


In [18]:
# 1. Check for duplicate event IDs (Primary Key integrity)
duplicate_ids = events_df.duplicated(subset=['id']).sum()
print(f"Duplicate Event IDs found: {duplicate_ids}")

# 2. Referential Integrity: Does every event belong to the match we pulled?
invalid_matches = events_df[events_df['match_id'] != match_id].shape[0]
print(f"Events linked to wrong match_id: {invalid_matches}")

# 3. Critical Null Values
critical_cols = ['type', 'minute', 'second', 'team', 'player', 'location']
print("\nNull values in critical columns:")
print(events_df[critical_cols].isnull().sum())

Duplicate Event IDs found: 0
Events linked to wrong match_id: 0

Null values in critical columns:
type         0
minute       0
second       0
team         0
player      29
location    55
dtype: int64


In [19]:
# Extract X and Y coordinates from the 'location' list
# We use dropna() because not all events (like substitutions) have coordinates
spatial_events = events_df.dropna(subset=['location']).copy()

# Split the [x, y] list into two separate flat columns
spatial_events[['x', 'y']] = pd.DataFrame(spatial_events['location'].tolist(), index=spatial_events.index)

print("\n--- Spatial Boundaries Check (Should be X: 0-120, Y: 0-80) ---")
display(spatial_events[['x', 'y']].describe())

# Check Expected Goals (xG) limits
shots = events_df[events_df['type'] == 'Shot'].copy()
if 'shot_statsbomb_xg' in shots.columns:
    print("\n--- Expected Goals (xG) Check (Should be 0.0 to 1.0) ---")
    display(shots['shot_statsbomb_xg'].describe())
else:
    print("No xG data available in this dataset.")


--- Spatial Boundaries Check (Should be X: 0-120, Y: 0-80) ---


,x,y
count,4352.000000,4352.000000
mean,57.891314,38.417739
std,28.084752,23.605068
min,1.000000,0.100000
25%,38.800000,17.000000
50%,55.700000,37.400000
75%,78.800000,59.600000
max,120.000000,80.000000



--- Expected Goals (xG) Check (Should be 0.0 to 1.0) ---


count    38.000000
mean      0.297340
std       0.327581
min       0.010578
25%       0.040560
50%       0.105311
75%       0.783500
max       0.783500
Name: shot_statsbomb_xg, dtype: float64

In [20]:
# Generate a professional automated HTML report
print("Generating Profiling Report. This may take a couple of minutes...")

# We filter out highly complex nested dictionary columns for the general report to avoid memory crashes
clean_df_for_report = events_df.select_dtypes(exclude=['object'])

profile = ProfileReport(
    events_df, 
    title="StatsBomb Event Data Quality Report",
    explorative=True,
    minimal=True # Set to False for a deeper, more computationally heavy analysis
)

# Save to your reports directory
profile.to_file("/Users/davidbazalduamendez/Documents/GitHub/100-days-Data-Sports-Challenge/reports/html/statsbomb_profiling_report.html")
print("Report saved to reports/statsbomb_profiling_report.html")

Generating Profiling Report. This may take a couple of minutes...


Export report to file: 100%|██████████| 1/1 [00:00<00:00, 148.63it/s]

Report saved to reports/statsbomb_profiling_report.html


In [21]:
print("--- Advanced Data Cleaning: The 3 Strategies for NaNs ---")

# STRATEGY 1: The "Implicit False" (Boolean Columns)
# StatsBomb uses NaN to mean 'False' for events that didn't happen. 
# For example, if a player wasn't under pressure, 'under_pressure' is NaN, not False.
boolean_cols = ['under_pressure', 'foul_committed_advantage', 'foul_won_advantage', 'pass_cross', 'shot_first_time']

# Filter only the columns that actually exist in this specific match's dataframe
cols_to_fill = [col for col in boolean_cols if col in events_df.columns]

# Impute (fill) with False
events_df[cols_to_fill] = events_df[cols_to_fill].fillna(False)
print(f"Strategy 1 Applied: Imputed 'False' for {len(cols_to_fill)} boolean columns.")


# STRATEGY 2: The "Safe Drop" (Noise / Ultra-Rare Events)
# Some columns are 99% empty and we won't use them for our models (e.g., 'injury_stoppage_in_match').
# We drop columns that are more than 98% empty to save RAM and clean our dataset.
threshold = 0.98
cols_to_drop = events_df.columns[events_df.isnull().mean() > threshold]

events_df = events_df.drop(columns=cols_to_drop)
print(f"Strategy 2 Applied: Dropped {len(cols_to_drop)} columns that were almost entirely NaNs.")


# STRATEGY 3: Contextual Nulls (Leave them alone!)
# We DO NOT fill 'pass_length' with 0.0 for a 'Shot' event. That would ruin our average pass length math!
# We leave them as NaNs. We only analyze them when we filter by the correct event type.
passes_df = events_df[events_df['type'] == 'Pass'].copy()
null_passes = passes_df['pass_length'].isnull().sum()

print(f"Strategy 3 Applied: Verified contextual nulls. ")
print(f"   -> Out of {len(passes_df)} passes, only {null_passes} have a NaN 'pass_length'.")

# Final check of our dataset shape
print(f"\nFinal cleaned dataset shape: {events_df.shape}")

--- Advanced Data Cleaning: The 3 Strategies for NaNs ---
Strategy 1 Applied: Imputed 'False' for 5 boolean columns.
Strategy 2 Applied: Dropped 56 columns that were almost entirely NaNs.
Strategy 3 Applied: Verified contextual nulls. 
   -> Out of 1263 passes, only 0 have a NaN 'pass_length'.

Final cleaned dataset shape: (4407, 38)
